# 14e · 把地图用到一个真实的药上

> **说明**：这一章用的标签是**细胞实验**的结果（药厂在 HEK 细胞里逐个测出来的），用来当机器学习的基准数据。它不是临床决策规则，这一章也不做临床解读。

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → [第二阶段 F：一条线，从一个分数到两个能量]
                 ├─ 14a 把稳定性变成能量 ✅
                 ├─ 14b 把测序读数变成两种测量 ✅
                 ├─ 14c 把一个分数拆成两个能量 ✅
                 ├─ 14d 把两个能量变成一张位点地图 ✅
                 ├─ 14e 把地图用到一个真实的药上  ← 你在这里
                 └─ 14f 把整条线讲出来 ✅

附录（不在这条线上）
                 └─ 14x 实验室该测多少个 ✅
```

## 背景：为什么这一章是整条线的落点

14c 把一个分数拆成了两个能量，14d 说"折叠坏了可以试试扶住它的小分子"。这一章**真的去验一次**。

**药理伴侣**（pharmacological chaperone）是一类很特别的药：它不抑制蛋白，而是在蛋白折叠的时候抱住它，让它别散架、别被细胞当垃圾处理掉。所以它只对一种突变有用——**蛋白该有的功能还在，只是折不好**。

法布里病（Fabry disease）是这件事做得最深的地方。药叫 **migalastat**（商品名 Galafold），2016–2018 年在欧美获批。获批前，药厂在细胞里**一个一个变体地测**了它对哪些 GLA 突变有效，并且把**有效和无效的名单都公开了**（Benjamin 等 2017）。

这是一份很难得的数据：

| 为什么难得 |
|---|
| **两边都有**。大部分药物数据只公布"有效"的那一半，没法训练也没法评估 |
| **GLA 对我们完全干净**。它不在任何训练或考试切分里，和训练集里的蛋白也不同源 |
| **没有已发表的 AUROC 可比**。做过这个题的最接近的一篇（IJMS 2025）只报了 p 值，没给分类指标 |

## 这一章做什么

| 步骤 | 内容 |
|---|---|
| 1 | 把名单从 97 页的补充材料 PDF 里解析出来 |
| 2 | 给每个 GLA 突变预测两个能量 |
| 3 | 检验："能被救回"对应哪一种能量的模式 |
| 4 | 和基线比，报告 AUROC 和置信区间 |

---

## 准备

In [ ]:
import os, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from databricks.sdk import WorkspaceClient
from qafi_platform import tracking
from qafi_platform.lakehouse import sql
from qafi_core import chaperone, energy, stability, validation
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"
WORK, MODELS, FABRY = ROOT / "data/stability", ROOT / "data/models", ROOT / "data/fabry"
FABRY.mkdir(parents=True, exist_ok=True)
GLA = "P06280"

---

## 第 1 步 · 把名单从 PDF 里挖出来

### 在干什么
Benjamin 等 2017 的补充材料是一个 97 页的 PDF，第 68 页起是"可救回"的表，第 77 页起是"不可救回"的表。把它们变成一张带标签的表。

### 背景：这类数据的常态
药物研发的关键数据经常以这种形式存在——**PDF 里的表格**。能可靠地把它变成数据，是一项实际技能。

两个坑，都踩过了：

| 坑 | 现象 | 怎么处理 |
|---|---|---|
| **目录页也有同样的标题** | 用标题切分，会把整份文件都切进第二张表，于是所有变体都被标成"不可救回" | 只认带剂量说明的标题（`Amenable GLA Mutations (`），目录页那行没有括号 |
| **每一页都重复表头** | 用"最后一次出现"切，又会切到最后一页 | 用**第一次**出现带括号的标题 |

还有两类行要排掉：**移码、剪接、截断**（它们没有"一个氨基酸换一个氨基酸"的写法），以及**终止密码子**（写成 X）。

### 执行
PDF 用 `pdftotext -layout` 转成文本（保留列对齐），解析代码在 `qafi_core.chaperone`，有 5 个测试盯着上面那些坑。

In [ ]:
pdf = FABRY / "benjamin2017_supplement.pdf"
if not pdf.exists():
    run(f"curl -sL -o {pdf} https://discovery.ucl.ac.uk/id/eprint/1519980/8/"
        "Benjamin_Validation_pharmacogenetics_S1.pdf")
    run(f"pdftotext -layout {pdf} {FABRY / 'benjamin2017.txt'}")
measured = chaperone.amenability((FABRY / "benjamin2017.txt").read_text())
contradictory = measured.attrs["listed_twice"]
measured = measured[~measured.variant.isin(contradictory)]
print(f"{len(measured)} 个单氨基酸替换，有干净标签："
      f"{measured.responds.sum()} 个可救回、{(~measured.responds).sum()} 个不可救回")
print(f"两张表里都出现、因而丢掉的: {len(contradictory)} 个 {contradictory}")
print("论文正文：600 个突变测试，268 可救回 / 332 不可救回（含移码、剪接等非替换类型）")
measured.head(4)

### 结果怎么看
**555 个**单氨基酸替换有干净标签：254 个可救回、301 个不可救回。

和论文正文的 268 / 332 对得上——差额是非替换类型（移码、剪接、截断）和终止密码子，它们没有"一个氨基酸换一个氨基酸"的形式。

有 **9 个**变体在两张表里都出现了。这是源数据本身的矛盾（同一个蛋白变化来自不同的核苷酸改变，测出了不同结果）。这一章把它们**全部丢掉**，不猜。

---

## 第 2 步 · 预测 GLA 的两个能量

### 在干什么
用 14c 的模型，给这 555 个突变各算一个折叠能和一个功能能。

### GLA 对模型有多干净
查过了：

- **不在任何切分里**——它没有深度突变扫描数据，07a 分训练/考试时根本没有它
- **和训练集里的任何蛋白都不同源**（用 07a 同一把尺子逐个比对过）

所以这是一次真正的"没见过"的预测。

### 执行

In [ ]:
splits = stability.load_splits(WORK / "mega_splits.pkl")
mega = stability.add_features(
    stability.megascale(WORK / "Tsuboyama2023_Dataset2_Dataset3_20230416.csv", splits),
    pd.read_parquet(WORK / "structure.parquet"), pd.read_parquet(WORK / "residue_index.parquet"),
    np.load(WORK / "logprobs.npy"))
mega = mega[mega.split.isin(["train", "val"])]
sequences = pd.read_parquet(MODELS / "proteome_sequences.parquet").set_index("uniprot_id").sequence
split_table = pd.read_csv(ROOT / "dbt/seeds/protein_split.csv")
locked = [sequences[u] for u in split_table.loc[split_table.split == "test", "uniprot_id"] if u in sequences.index]
mega = mega[~mega.domain.isin({d for d, s in mega.groupby("domain").seq.first().items()
                               if any(validation.related(s, t) for t in locked)})]
print("GLA 在切分表里吗:", GLA in set(split_table.uniprot_id))
print("GLA 的同源训练蛋白:",
      [u for u in split_table.loc[split_table.split == 'train', 'uniprot_id']
       if u in sequences.index and validation.related(sequences[GLA], sequences[u])] or "没有")
labels = pd.read_parquet(MODELS / "train_variants.parquet")
titles = sql(w, "SELECT dataset_id, title FROM qafi_dev.gold.dim_label_dataset").set_index("dataset_id").title.to_dict()
labels["reads"] = [energy.readout_kind(k, titles.get(d))
                   for k, d in zip(labels.data_kind, labels.dataset_id, strict=True)]
kinds = labels.groupby("uniprot_id").reads.apply(set)
paired = {p for p, k in kinds.items() if {energy.ABUNDANCE, energy.ACTIVITY} <= k}
rows, vectors = energy.stack(mega, np.load(WORK / "vectors.npy"), labels,
                             pd.read_parquet(MODELS / "train_embeddings.parquet"),
                             np.asarray(np.load(MODELS / "train_embeddings.npy", mmap_mode="r")), titles)
models = [energy.fit(rows[(rows.kind == energy.DDG) | rows.protein.isin(paired)], vectors,
                     energy.FEATURES, seed=s) for s in (0, 1, 2)]
print("模型训练完成")

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "databricks" / "src"))
from lakehouse import language_model as lm
sequence = sequences[GLA]
residues = pd.read_parquet(MODELS / "proteome_residues.parquet",
                           filters=[("uniprot_id", "==", GLA)]).sort_values("position")
model, tokenizer = lm.load()
logprobs, gla_vectors = lm.read_proteins([sequence], model, tokenizer, embeddings=True)[0]
index = pd.DataFrame({"domain": GLA, "position": np.arange(1, len(sequence) + 1)})
structure = residues[["uniprot_id", "position", "aa", "plddt", "rsa", "contacts_8", "contacts_12",
                      "hse_up", "hse_down"]].rename(columns={"uniprot_id": "domain"})
mismatch = [v.variant for v in measured.itertuples()
            if v.position > len(sequence) or sequence[v.position - 1] != v.ref_aa]
print(f"参考氨基酸和 UniProt 对不上的: {len(mismatch)}")
variants = measured.assign(domain=GLA, wt_aa=measured.ref_aa, mut_aa=measured.alt_aa, ddg=0.0)
variants = stability.add_features(variants, structure, index, logprobs, strict=True)
variants["row"] += len(vectors)
every = np.concatenate([vectors, gla_vectors.astype(np.float16)])
fold, func = zip(*(energy.energies(m, s, variants, every, energy.FEATURES) for m, s, _ in models))
variants["ddg_fold"], variants["ddg_func"] = np.mean(fold, axis=0), np.mean(func, axis=0)
print(f"{len(variants)} 个突变打完分")
variants.groupby("responds")[["ddg_fold", "ddg_func"]].median().round(3)

### 结果怎么看

| | 折叠能中位数 | 功能能中位数 |
|---|---|---|
| **能被救回** | −0.81 | **−0.14** |
| 救不回 | −1.35 | **−0.58** |

两个能量都显示"能被救回的变体伤得更轻"，但**功能能的差距大得多**（−0.14 对 −0.58，差了 0.44 kcal/mol），折叠能的差距小一些（0.54，但基数大）。

**这和我事先的猜想不一样。** 我原本按 14d 的说法预期"折叠坏了但功能完好 = 能救回"。数据说的是：**能救回的是那些折叠只是轻微受损、而且活性位点完好的变体。**

想想 migalastat 的机理，这其实更合理：它是一个**结合在活性位点上**的小分子（一种亚氨基糖）。它要起作用，活性位点本身必须还在；而且如果蛋白被破坏得太彻底，扶也扶不起来。所以"轻度失稳 + 活性位点完好"才是可救回的样子。

---

## 第 3 步 · 能分得出来吗

### 在干什么
用 AUROC 衡量：按某个分数排序，能不能把"可救回"的排在前面。0.5 = 瞎猜，1.0 = 完美。

### 对照
没有已发表的 AUROC 可比（这个题目基本是空白），所以和通用的预测器比：

| 对照 | 是什么 |
|---|---|
| **ESM-2 对数概率差** | 蛋白语言模型的零样本分，最常用的通用基准 |
| **埋藏 × 保守性** | 14a 用过的零参数基线 |
| **溶剂可及度** | 只看这个位置埋得深不深 |

### 执行

In [ ]:
y = variants.responds.to_numpy()
single = {
    "我们的功能能": variants.ddg_func,
    "我们的折叠能": variants.ddg_fold,
    "ESM-2 对数概率差": variants.esm_delta,
    "埋藏 × 保守性": variants.rsalor,
    "溶剂可及度": variants.rsa,
}
rng = np.random.default_rng(0)
def interval(values):
    draws = [roc_auc_score(y[i], np.asarray(values)[i])
             for i in (rng.integers(0, len(y), len(y)) for _ in range(1000)) if len(set(y[i])) == 2]
    return np.percentile(draws, 2.5), np.percentile(draws, 97.5)
out = []
for name, values in single.items():
    low, high = interval(values)
    out.append({"分数": name, "AUROC": roc_auc_score(y, values), "AUPRC": average_precision_score(y, values),
                "95% 区间": f"{low:.3f}–{high:.3f}"})
pd.DataFrame(out).sort_values("AUROC", ascending=False, ignore_index=True).round(3)

### 结果怎么看

| 分数 | AUROC | AUPRC | 95% 区间 |
|---|---|---|---|
| **我们的功能能** | **0.759** | 0.650 | 0.720–0.799 |
| ESM-2 对数概率差 | 0.729 | 0.646 | 0.686–0.772 |
| 埋藏 × 保守性 | 0.715 | 0.597 | 0.668–0.758 |
| 我们的折叠能 | 0.693 | 0.629 | 0.652–0.737 |
| 溶剂可及度 | 0.631 | 0.539 | 0.584–0.674 |

**功能能单独就是最好的单一分数**（0.759），超过 ESM-2 的 0.729，而且两者的 95% 区间几乎不重叠。

折叠能本身也有 0.693，方向是"**折叠伤得越轻越容易被救回**"——伤太重的扶不起来。两个能量指向同一件事的两个侧面。

两个数一起会不会更好？下面用**按位置分折**的交叉验证来看（同一个位置的不同替换不能跨折，否则就是泄漏）。

In [ ]:
sets = {"两个能量": ["ddg_fold", "ddg_func"],
        "两个能量 + 埋藏": ["ddg_fold", "ddg_func", "rsa"],
        "ESM-2 + 两个能量": ["esm_delta", "ddg_fold", "ddg_func"],
        "ESM-2": ["esm_delta"],
        "ESM-2 + 埋藏": ["esm_delta", "rsa"]}
combined = []
for name, columns in sets.items():
    predicted = np.zeros(len(variants))
    for train, test in GroupKFold(n_splits=5).split(variants, y, groups=variants.position):
        fitted = LogisticRegression(max_iter=2000).fit(variants.iloc[train][columns], y[train])
        predicted[test] = fitted.predict_proba(variants.iloc[test][columns])[:, 1]
    combined.append({"特征": name, "AUROC": roc_auc_score(y, predicted),
                     "AUPRC": average_precision_score(y, predicted)})
pd.DataFrame(combined).sort_values("AUROC", ascending=False, ignore_index=True).round(3)

### 结果怎么看

| 特征 | AUROC | AUPRC |
|---|---|---|
| **两个能量 + 埋藏** | **0.769** | **0.684** |
| 两个能量 | 0.764 | 0.646 |
| ESM-2 + 两个能量 | 0.763 | 0.649 |
| ESM-2 | 0.725 | 0.636 |
| ESM-2 + 埋藏 | 0.721 | 0.633 |

**两个能量合起来 0.769，比 ESM-2 的 0.725 高 0.044。** 而且加上 ESM-2 并不会让两个能量更好（0.763 < 0.764）——说明能量已经把 ESM-2 的信息包含进去了，还多了点别的。

这个差距不算大，但方向明确，而且是在一个**模型完全没见过的蛋白**上、用**位置分折**的交叉验证得到的。

### 这一章真正的意义

AUROC 0.77 不会拿去指导任何临床决定。它的意义在别处：**14c 拆出来的那个"功能能"，在一个全新的蛋白上、对一个真实的药物问题，比通用的单分数更有用。** 这就把整条线闭上了：

```
14c 拆出两个能量 → 14d 说哪些位置是"结构坏/功能坏" → 14e 证明这个区分对选药物类型真的有信息
```

### 面试怎么说
> "Migalastat is an approved pharmacological chaperone and its makers published which GLA variants respond and which do not — both classes, which is rare. On that set, with GLA held out of everything and position-grouped cross-validation, the two energies reach AUROC 0.769 against 0.725 for a zero-shot language model. The functional energy alone is the best single score. The direction on the folding energy was the opposite of my prior and makes sense: the drug binds the active site, so it rescues mildly destabilised variants with intact machinery, not severely broken ones."

---

## 第 4 步 · 记账

In [ ]:
best = pd.DataFrame(combined).set_index("特征")
metrics = {"auroc_two_energies": float(best.loc["两个能量", "AUROC"]),
           "auroc_two_energies_and_burial": float(best.loc["两个能量 + 埋藏", "AUROC"]),
           "auroc_esm2": float(best.loc["ESM-2", "AUROC"]),
           "auroc_functional_energy_alone": float(roc_auc_score(y, variants.ddg_func)),
           "variants": len(variants), "rescued": int(y.sum())}
tracking.experiment_run(
    EXPERIMENT, "chaperone_rescue",
    params={"protein": GLA, "labels": "Benjamin 2017 GLP HEK assay", "cv": "GroupKFold by position",
            "readout_proteins": len(paired), "dropped_contradictory": len(contradictory)},
    metrics=metrics, tables={"single": pd.DataFrame(out), "combined": pd.DataFrame(combined)},
    chapter="14e")
print("已记录"); pd.Series(metrics)

---

## 这一章做完了什么

- 从一个 97 页的补充材料 PDF 里解析出 **555 个带标签的 GLA 替换**（254 可救回 / 301 不可救回），避开了目录页和重复页眉两个坑，9 个自相矛盾的条目丢掉
- 在一个**完全没见过、也没有同源亲戚**的蛋白上预测两个能量
- **功能能是最好的单一分数**（AUROC 0.759），两个能量合起来 **0.769**，超过 ESM-2 零样本的 0.725（按位置分折）
- 纠正了一个猜错的方向：药理伴侣救的是**轻度失稳 + 活性位点完好**的变体，不是"折叠全坏但功能完好"的——这和 migalastat 结合活性位点的机理吻合

## 下一步

**14f · 把整条线讲出来**：主线六章做完了。最后一章把 README 改写成招聘方看得懂的样子，按三类岗位各写一段讲稿和简历条目。

## 提交

In [ ]:
run("git status --short")

In [ ]:
run("git add -A && git commit -q -m 'feat: 14e which variants a pharmacological chaperone rescues' && git log --oneline -1")